# Discover → custom assembly → move (controls)

Build profiles in `config` terms for HostProxy / scripts — not the pcb_lab TUI.
For a remote command→FB prove, prefer `closed_loop_smoke.ipynb`.

One COM owner. Close board-verify / dashboard first. Kernel cwd: `scripts/` or `scripts/notebooks/`.

In [ ]:
import os, sys

cwd = os.path.abspath(os.getcwd())
if os.path.basename(cwd) == "notebooks":
    SCRIPTS = os.path.abspath(os.path.join(cwd, ".."))
elif os.path.isdir(os.path.join(cwd, "deft_controls_sdk")):
    SCRIPTS = cwd
else:
    SCRIPTS = os.path.abspath(os.path.join(cwd, "scripts"))
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)

from deft_controls_sdk import HostProxy
from deft_controls_sdk.config import (
    Assembly,
    assembly_put_actuator,
    single_profile,
    wheel_profile,
    arm_profile,
    CfgSlotSpec,
    PROTO_ROBSTRIDE,
    PROTO_DAMIAO,
    assembly_from_name,
)

PORT = "COM5"  # change me
asm0 = assembly_from_name("bench")
proxy = HostProxy.connect(PORT, mode="debug", armed=False, assembly=asm0)
hub = proxy.hub
print(proxy.doctor().get("ok"), hub.port, proxy.mode)

## 1. Discover

Pick the call that matches the bus. Use tight ID ranges.

In [ ]:
# RobStride on CH5/CH6 (bench wheels)
rs = hub.debug.discover_robstride_by_bus(buses=[5, 6], start=0x70, end=0x75)
print("RS", rs)

# Damiao on CH1 (left arm)
dm = hub.debug.discover_damiao_all(bus=1, start=1, end=8)
print("DM bus1", dm)

# Or one queued pass:
# hits = hub.debug.discover_queued(
#     buses=[1, 5, 6],
#     protocols=("robstride", "damiao"),
#     ranges={"robstride": (0x70, 0x75), "damiao": (1, 8)},
# )
# print(hits)

## 2. Build a custom assembly

Map discovered `(bus, protocol, motor_id)` → host **slots**. Edit the lists to match what you found.

In [ ]:
# Example: 4 RS wheels on spare slots 22–25 (edit ids from discover)
base = wheel_profile(name="base", slots=(22, 23, 24, 25)).with_cfg((
    CfgSlotSpec(bus=5, protocol=PROTO_ROBSTRIDE, motor_id=0x70),
    CfgSlotSpec(bus=5, protocol=PROTO_ROBSTRIDE, motor_id=0x71),
    CfgSlotSpec(bus=6, protocol=PROTO_ROBSTRIDE, motor_id=0x72),
    CfgSlotSpec(bus=6, protocol=PROTO_ROBSTRIDE, motor_id=0x73),
))

# Example: left arm product map (or build from dm hits)
left = arm_profile("yam", side="left")  # slots 0–6 + YAM CFG

# Or one motor:
# m = single_profile(22, protocol="robstride", motor_id=0x70, bus=5, name="m0")

asm = Assembly(name="lab_custom", actuators={}, servos={})
asm = assembly_put_actuator(asm, base)
asm = assembly_put_actuator(asm, left)

for n, p in asm.actuators.items():
    print(n, list(p.slots), p.as_cfg_rows())

## 3. Apply CFG to the board, rebind LabRobot

`persist=True` writes flash; omit / `False` for RAM-only.

In [ ]:
for prof in asm.actuators.values():
    for row in prof.as_cfg_rows():
        hub.debug.cfg_set_slot(**row, persist=False)

proxy.close()
proxy = HostProxy.connect(PORT, mode="debug", armed=False, assembly=asm)
hub = proxy.hub
print("bound", list(asm.actuators), "armed", proxy.armed)

## 4. Move

Clear path. Small deltas. Always blank when done.

In [ ]:
import time

proxy.arm_plant()
a = proxy.actions
base = a.actuator(name="base")  # assembly bound at connect

a.mount(base.hold())  # sample FB → stay put
a.apply()
time.sleep(1.0)

a.mount(base.nudge(index=0, delta=0.05))
print("pending", a.pending)
a.apply()
time.sleep(1.0)

print("base FB", base.positions())
a.clear()
proxy.disarm_plant()

In [ ]:
proxy.close()  # release COM